In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW  = Path('../data/raw')
PROC = Path('../data/processed')
PROC.mkdir(exist_ok=True)

# Load all tables
print("Loading tables")
train        = pd.read_csv(RAW / 'train_v2.csv')
members      = pd.read_csv(RAW / 'members_v3.csv')
transactions = pd.read_csv(RAW / 'transactions_v2.csv')

print(f"Train:        {train.shape}")
print(f"Members:      {members.shape}")
print(f"Transactions: {transactions.shape}")

Loading tables
Train:        (970960, 2)
Members:      (6769473, 6)
Transactions: (1431009, 9)


In [3]:
# Cleaning Members csv file

# 1. Filter to training users only
members = members[members['msno'].isin(train['msno'])].copy()
print(f"Members filtered to training users: {members.shape}")

# 2. Age cleaning
members['age_known'] = ((members['bd'] >= 10) & (members['bd'] <= 70)).astype(int)
members['age'] = members['bd'].where((members['bd'] >= 10) & (members['bd'] <= 70), np.nan)
print(f"\nAge known: {members['age_known'].mean():.1%}")

# 3. Gender encoding
members['gender_clean'] = members['gender'].fillna('unknown')
gender_dummies = pd.get_dummies(members['gender_clean'], prefix='gender').astype(int)
members = pd.concat([members, gender_dummies], axis=1)
print(f"Gender distribution:\n{members['gender_clean'].value_counts()}")

# 4. Registration date → days since registration
members['registration_init_time'] = pd.to_datetime(
    members['registration_init_time'].astype(str), format='%Y%m%d', errors='coerce'
)
reference_date = pd.Timestamp('2017-04-01')  # day after log data ends
members['days_since_registration'] = (reference_date - members['registration_init_time']).dt.days
print(f"\nDays since registration — mean: {members['days_since_registration'].mean():.0f}, "
      f"min: {members['days_since_registration'].min()}, "
      f"max: {members['days_since_registration'].max()}")

# 5. Keep only useful columns
members_clean = members[[
    'msno', 'city', 'age', 'age_known',
    'gender_female', 'gender_male', 'gender_unknown',
    'registered_via', 'days_since_registration'
]].copy()

print(f"\nCleaned members shape: {members_clean.shape}")
members_clean.head()

Members filtered to training users: (860967, 6)

Age known: 44.8%
Gender distribution:
gender_clean
unknown    472062
male       204561
female     184344
Name: count, dtype: int64

Days since registration — mean: 1296, min: -23, max: 4754

Cleaned members shape: (860967, 9)


,msno,city,age,age_known,gender_female,gender_male,gender_unknown,registered_via,days_since_registration
1,+tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=,1,NaN,0,0,0,1,7,2026
5,yLkV2gbZ4GLFwqTOXLVHz0VGrMYcgBGgKZ3kj9RiYu8=,4,30.0,1,0,1,0,9,2024
9,I0yFvqMoNkM8ZNHb617e1RBzIS/YRKemHO7Wj13EtA0=,13,63.0,1,0,1,0,9,2022
10,OoDwiKZM+ZGr9P3fRivavgOtglTEaNfWJO4KaJcTTts=,1,NaN,0,0,0,1,7,2022
13,4De1jAxNRABoyRBDZ82U0yEmzYkqeOugRGVNIf92Xb8=,4,28.0,1,1,0,0,9,2020


In [4]:
# Transaction CSV file

# Convert dates
transactions['transaction_date'] = pd.to_datetime(
    transactions['transaction_date'].astype(str), format='%Y%m%d', errors='coerce'
)
transactions['membership_expire_date'] = pd.to_datetime(
    transactions['membership_expire_date'].astype(str), format='%Y%m%d', errors='coerce'
)

# Filter to training users only
transactions = transactions[transactions['msno'].isin(train['msno'])].copy()
print(f"Transactions for training users: {transactions.shape}")

# Discount flag
transactions['had_discount'] = (
    transactions['plan_list_price'] - transactions['actual_amount_paid'] > 0
).astype(int)

# Aggregate per user
txn_features = transactions.groupby('msno').agg(
    txn_count             = ('transaction_date', 'count'),
    cancel_count          = ('is_cancel', 'sum'),
    ever_cancelled        = ('is_cancel', 'max'),
    auto_renew_rate       = ('is_auto_renew', 'mean'),
    avg_plan_days         = ('payment_plan_days', 'mean'),
    most_common_plan_days = ('payment_plan_days', lambda x: x.mode()[0]),
    avg_price_paid        = ('actual_amount_paid', 'mean'),
    total_paid            = ('actual_amount_paid', 'sum'),
    discount_rate         = ('had_discount', 'mean'),
    unique_payment_methods= ('payment_method_id', 'nunique'),
    last_transaction_date = ('transaction_date', 'max'),
    membership_expire_date= ('membership_expire_date', 'max'),
).reset_index()

# Days until membership expires from reference date
reference_date = pd.Timestamp('2017-04-01')
txn_features['days_until_expiry'] = (
    txn_features['membership_expire_date'] - reference_date
).dt.days
txn_features['days_since_last_txn'] = (
    reference_date - txn_features['last_transaction_date']
).dt.days

# Cancel ratio
txn_features['cancel_ratio'] = (
    txn_features['cancel_count'] / txn_features['txn_count']
)

# Drop raw date columns
txn_features = txn_features.drop(
    columns=['last_transaction_date', 'membership_expire_date']
)

print(f"Transaction features shape: {txn_features.shape}")
print(f"\nSample stats:")
print(txn_features[['txn_count','cancel_count','auto_renew_rate',
                     'avg_plan_days','days_until_expiry']].describe().round(2))
txn_features.head()

Transactions for training users: (1132036, 9)
Transaction features shape: (933578, 14)

Sample stats:
       txn_count  cancel_count  auto_renew_rate  avg_plan_days  \
count  933578.00     933578.00        933578.00      933578.00   
mean        1.21          0.04             0.91          33.67   
std         1.29          0.19             0.28          31.54   
min         1.00          0.00             0.00           0.00   
25%         1.00          0.00             1.00          30.00   
50%         1.00          0.00             1.00          30.00   
75%         1.00          0.00             1.00          30.00   
max       208.00          8.00             1.00         450.00   

       days_until_expiry  
count          933578.00  
mean               24.17  
std                49.69  
min               -32.00  
25%                 8.00  
50%                18.00  
75%                26.00  
max              2329.00  


,msno,txn_count,cancel_count,ever_cancelled,auto_renew_rate,avg_plan_days,most_common_plan_days,avg_price_paid,total_paid,discount_rate,unique_payment_methods,days_until_expiry,days_since_last_txn,cancel_ratio
0,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,1,0,0,1.0,30.0,30,99.0,99,0.0,1,14,17,0.0
1,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,2,0,0,1.0,30.0,30,149.0,298,0.0,1,48,1,0.0
2,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,1,0,0,1.0,30.0,30,149.0,149,0.0,1,25,6,0.0
3,++/9R3sX37CjxbY/AaGvbwr3QkwElKBCtSvVzhCBDOk=,1,0,0,1.0,30.0,30,149.0,149,0.0,1,14,17,0.0
4,++/UDNo9DLrxT8QVGiDi1OnWfczAdEwThaVyD0fXO50=,2,0,0,1.0,30.0,30,149.0,298,0.0,1,52,1,0.0


In [5]:
# ── User Log CSV file

from collections import defaultdict

print("Aggregating user logs in chunks...")

# Accumulators
agg = defaultdict(lambda: {
    'total_days'      : 0,
    'num_25'          : 0,
    'num_50'          : 0,
    'num_75'          : 0,
    'num_985'         : 0,
    'num_100'         : 0,
    'num_unq'         : 0,
    'total_secs'      : 0,
})

train_users = set(train['msno'])
chunk_size  = 500_000

for i, chunk in enumerate(pd.read_csv(RAW / 'user_logs_v2.csv',
                                       chunksize=chunk_size)):
    # Filter to training users only
    chunk = chunk[chunk['msno'].isin(train_users)]

    for row in chunk.itertuples(index=False):
        u = agg[row.msno]
        u['total_days'] += 1
        u['num_25']     += row.num_25
        u['num_50']     += row.num_50
        u['num_75']     += row.num_75
        u['num_985']    += row.num_985
        u['num_100']    += row.num_100
        u['num_unq']    += row.num_unq
        u['total_secs'] += row.total_secs

    if i % 5 == 0:
        print(f"  Chunk {i+1} done — {len(agg):,} users accumulated")

# Convert to DataFrame
log_features = pd.DataFrame.from_dict(agg, orient='index').reset_index()
log_features.rename(columns={'index': 'msno'}, inplace=True)

# Derived features
total_songs = (log_features['num_25'] + log_features['num_50'] +
               log_features['num_75'] + log_features['num_985'] +
               log_features['num_100'])

log_features['avg_daily_songs']    = total_songs / log_features['total_days']
log_features['avg_daily_secs']     = log_features['total_secs'] / log_features['total_days']
log_features['avg_daily_unq']      = log_features['num_unq'] / log_features['total_days']
log_features['completion_rate']    = (log_features['num_100'] /
                                       total_songs.replace(0, np.nan))
log_features['skip_rate']          = (log_features['num_25'] /
                                       total_songs.replace(0, np.nan))

print(f"\nLog features shape: {log_features.shape}")
print(f"Users with log data: {len(log_features):,} "
      f"({len(log_features)/len(train_users):.1%} of training users)")
print(f"\nSample stats:")
print(log_features[['total_days','avg_daily_songs',
                     'avg_daily_secs','completion_rate',
                     'skip_rate']].describe().round(3))

Aggregating user logs in chunks...
  Chunk 1 done — 277,300 users accumulated
  Chunk 6 done — 642,343 users accumulated
  Chunk 11 done — 699,843 users accumulated
  Chunk 16 done — 722,767 users accumulated
  Chunk 21 done — 735,387 users accumulated
  Chunk 26 done — 743,527 users accumulated
  Chunk 31 done — 749,426 users accumulated
  Chunk 36 done — 753,900 users accumulated

Log features shape: (754551, 14)
Users with log data: 754,551 (77.7% of training users)

Sample stats:
       total_days  avg_daily_songs  avg_daily_secs  completion_rate  \
count  754551.000       754551.000      754551.000       754551.000   
mean       17.935           33.046        6471.711            0.704   
std         9.677           29.398        6419.716            0.206   
min         1.000            1.000           0.035            0.000   
25%         9.000           16.000        2858.342            0.591   
50%        19.000           25.333        4709.031            0.745   
75%        27.

In [7]:
# ── Master Feature Table

# Add has_log_data flag before joining
log_features['has_log_data'] = 1

# Left join everything onto train
df = train.copy()
df = df.merge(members_clean,  on='msno', how='left')
df = df.merge(txn_features,   on='msno', how='left')
df = df.merge(log_features,   on='msno', how='left')

# Fill missing log data flag
df['has_log_data'] = df['has_log_data'].fillna(0).astype(int)

print(f"Master table shape: {df.shape}")
print(f"\nMissing values after join:")
missing = df.isnull().sum()
print(missing[missing > 0])

Master table shape: (970960, 37)

Missing values after join:
city                       109993
age                        584828
age_known                  109993
gender_female              109993
gender_male                109993
gender_unknown             109993
registered_via             109993
days_since_registration    109993
txn_count                   37382
cancel_count                37382
ever_cancelled              37382
auto_renew_rate             37382
avg_plan_days               37382
most_common_plan_days       37382
avg_price_paid              37382
total_paid                  37382
discount_rate               37382
unique_payment_methods      37382
days_until_expiry           37382
days_since_last_txn         37382
cancel_ratio                37382
total_days                 216409
num_25                     216409
num_50                     216409
num_75                     216409
num_985                    216409
num_100                    216409
num_unq              

In [8]:
# Imputing Missing Values

# Member features (users not in members table)
member_cols_numeric = ['city', 'registered_via', 'days_since_registration']
member_cols_binary  = ['age_known', 'gender_female', 'gender_male', 'gender_unknown']

for col in member_cols_numeric:
    df[col] = df[col].fillna(df[col].median())

for col in member_cols_binary:
    df[col] = df[col].fillna(0)

# Age: fill missing with median of known ages
df['age'] = df['age'].fillna(df['age'].median())

# Clip negative days_since_registration
df['days_since_registration'] = df['days_since_registration'].clip(lower=0)

# Transaction features (users with no history)
txn_fill_zero = ['txn_count', 'cancel_count', 'ever_cancelled',
                 'total_paid', 'discount_rate', 'unique_payment_methods']
txn_fill_median = ['auto_renew_rate', 'avg_plan_days', 'most_common_plan_days',
                   'avg_price_paid', 'days_until_expiry', 'days_since_last_txn',
                   'cancel_ratio']

for col in txn_fill_zero:
    df[col] = df[col].fillna(0)

for col in txn_fill_median:
    df[col] = df[col].fillna(df[col].median())

# Clip negative days_until_expiry — already expired, set to 0
df['days_until_expiry'] = df['days_until_expiry'].clip(lower=0)

# ── Log features (users with zero activity
log_cols = ['total_days', 'num_25', 'num_50', 'num_75', 'num_985',
            'num_100', 'num_unq', 'total_secs', 'avg_daily_songs',
            'avg_daily_secs', 'avg_daily_unq', 'completion_rate', 'skip_rate']

for col in log_cols:
    df[col] = df[col].fillna(0)

# Cap extreme outliers in log features
df['avg_daily_secs']  = df['avg_daily_secs'].clip(upper=df['avg_daily_secs'].quantile(0.999))
df['avg_daily_songs'] = df['avg_daily_songs'].clip(upper=df['avg_daily_songs'].quantile(0.999))

# Final check
print(f"Shape: {df.shape}")
print(f"Missing values remaining: {df.isnull().sum().sum()}")
print(f"\nChurn rate: {df['is_churn'].mean():.2%}")
print(f"\nFeature dtypes:\n{df.dtypes.value_counts()}")

Shape: (970960, 37)
Missing values remaining: 0

Churn rate: 8.99%

Feature dtypes:
float64    34
int64       2
object      1
Name: count, dtype: int64


In [9]:
#Finalize and Save

# Define feature columns (drop msno and target)
feature_cols = [c for c in df.columns if c not in ['msno', 'is_churn']]

print(f"Total features: {len(feature_cols)}")
print(f"\nFeature list:")
for i, col in enumerate(feature_cols):
    print(f"  {i+1:2d}. {col}")

# Save master table
df.to_csv(PROC / 'features.csv', index=False)

# Save feature column list
import json
with open(PROC / 'feature_columns.json', 'w') as f:
    json.dump(feature_cols, f, indent=2)

print(f"\nSaved features.csv — {df.shape}")
print(f"Saved feature_columns.json — {len(feature_cols)} features")

# Quick churn profile — churners vs non-churners on key features
print("\n── Churner vs Non-churner profile ──")
key_cols = ['total_days', 'avg_daily_songs', 'completion_rate',
            'skip_rate', 'ever_cancelled', 'auto_renew_rate',
            'days_until_expiry', 'has_log_data']
print(df.groupby('is_churn')[key_cols].mean().round(3).T.to_string())

Total features: 35

Feature list:
   1. city
   2. age
   3. age_known
   4. gender_female
   5. gender_male
   6. gender_unknown
   7. registered_via
   8. days_since_registration
   9. txn_count
  10. cancel_count
  11. ever_cancelled
  12. auto_renew_rate
  13. avg_plan_days
  14. most_common_plan_days
  15. avg_price_paid
  16. total_paid
  17. discount_rate
  18. unique_payment_methods
  19. days_until_expiry
  20. days_since_last_txn
  21. cancel_ratio
  22. total_days
  23. num_25
  24. num_50
  25. num_75
  26. num_985
  27. num_100
  28. num_unq
  29. total_secs
  30. avg_daily_songs
  31. avg_daily_secs
  32. avg_daily_unq
  33. completion_rate
  34. skip_rate
  35. has_log_data

Saved features.csv — (970960, 37)
Saved feature_columns.json — 35 features

── Churner vs Non-churner profile ──
is_churn                0       1
total_days         14.097  12.327
avg_daily_songs    25.573  25.687
completion_rate     0.548   0.542
skip_rate           0.145   0.144
ever_cancelled    